### Análise de Inadimplência pessoa física(PF) por forma de pagamento.

## Fonte: Banco Central

In [ ]:
import pandas as pd
import requests
from datetime import datetime
import plotly.express as px
import plotly.io as pio
# pio.renderers.default = "vscode"
pio.renderers.default = "png"

In [28]:
def baixar_serie_bcb(codigo, nome, data_inicio="01/01/2011"):

    url = (
        f"https://api.bcb.gov.br/dados/serie/"
        f"bcdata.sgs.{codigo}/dados"
        f"?formato=json&dataInicial={data_inicio}"        
    )

    resposta = requests.get(url)
    resposta.raise_for_status()

    dados = resposta.json()
    df = pd.DataFrame(dados)
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
    df["valor"] = pd.to_numeric(df["valor"])
    df = df.rename(columns={"valor":nome})
    df = df.set_index("data")

    return df

In [29]:
series = {
    "inad_cheque_especial": 21113,
    "inad_cartao_rotativo": 21127,
    "inad_cartao_parcelado": 21128,
    "inad_consignado_privado": 21116,
    "inad_consignado_publico": 21117,
    "inad_pf_total": 21084,
}

In [30]:
dfs = []
for nome, codigo in series.items():
    print(f"Baixando {nome} (SGS {codigo})...")
    df = baixar_serie_bcb(codigo,nome)
    dfs.append(df)


Baixando inad_cheque_especial (SGS 21113)...
Baixando inad_cartao_rotativo (SGS 21127)...
Baixando inad_cartao_parcelado (SGS 21128)...
Baixando inad_consignado_privado (SGS 21116)...
Baixando inad_consignado_publico (SGS 21117)...
Baixando inad_pf_total (SGS 21084)...


In [31]:
base = pd.concat(dfs, axis=1)


In [32]:
base

,inad_cheque_especial,inad_cartao_rotativo,inad_cartao_parcelado,inad_consignado_privado,inad_consignado_publico,inad_pf_total
data,,,,,,
2011-03-01,10.82,30.06,1.58,4.74,2.97,4.62
2011-04-01,10.97,29.18,2.07,4.74,2.98,4.67
2011-05-01,10.95,31.80,2.44,4.78,2.98,4.82
2011-06-01,10.84,32.54,3.02,4.85,3.00,4.79
2011-07-01,11.60,34.73,3.23,4.94,3.03,4.90
...,...,...,...,...,...,...
2026-04-01,15.55,60.61,12.71,7.45,2.59,5.51
2026-05-01,15.92,63.07,12.49,7.95,2.62,5.63
2026-06-01,16.26,62.65,10.47,8.68,2.58,5.42


In [39]:
selic = pd.read_excel("selic_mensal.xlsx")

selic.set_index("data")

,selic_mensal
data,
2011-03-01,0.92
2011-04-01,0.84
2011-05-01,0.99
2011-06-01,0.96
2011-07-01,0.97
...,...
2026-06-01,1.12
2026-07-01,1.22
2026-08-01,1.09


In [42]:
base_final = pd.merge(base, selic, on="data")

base_final
base_final.dropna()

,data,inad_cheque_especial,inad_cartao_rotativo,inad_cartao_parcelado,inad_consignado_privado,inad_consignado_publico,inad_pf_total,selic_mensal
0,2011-03-01,10.82,30.06,1.58,4.74,2.97,4.62,0.92
1,2011-04-01,10.97,29.18,2.07,4.74,2.98,4.67,0.84
2,2011-05-01,10.95,31.80,2.44,4.78,2.98,4.82,0.99
3,2011-06-01,10.84,32.54,3.02,4.85,3.00,4.79,0.96
4,2011-07-01,11.60,34.73,3.23,4.94,3.03,4.90,0.97
...,...,...,...,...,...,...,...,...
181,2026-04-01,15.55,60.61,12.71,7.45,2.59,5.51,1.09
182,2026-05-01,15.92,63.07,12.49,7.95,2.62,5.63,1.07
183,2026-06-01,16.26,62.65,10.47,8.68,2.58,5.42,1.12
184,2026-07-01,16.31,65.55,11.21,10.00,2.69,5.80,1.22


In [71]:
print("Shape:", base_final.shape)
print("Período:", base_final.index.min(), "a", base_final.index.max())
print("\nValores nulos:")
print(base_final.isnull().sum())
print("\nEstatísticas descritivas:")
display(base_final.describe().round(2))

Shape: (186, 8)
Período: 0 a 185

Valores nulos:
data                  0
cheque_especial       0
cartao_rotativo       0
cartao_parcelado      0
consignado_privado    0
consignado_publico    0
pf_total              0
selic_mensal          0
dtype: int64

Estatísticas descritivas:


,data,cheque_especial,cartao_rotativo,cartao_parcelado,consignado_privado,consignado_publico,pf_total,selic_mensal
count,186,186.00,186.00,186.00,186.00,186.00,186.00,186.00
mean,2018-11-15 18:11:36.774193,13.74,41.31,4.89,5.17,2.44,4.10,0.79
min,2011-03-01 00:00:00,9.44,28.79,0.33,3.79,1.87,2.89,0.13
25%,2015-01-08 18:00:00,12.53,34.92,1.59,4.36,2.20,3.66,0.55
50%,2018-11-16 00:00:00,13.68,36.99,4.21,5.02,2.34,4.00,0.84
75%,2022-09-23 12:00:00,15.21,46.94,6.80,5.51,2.70,4.37,1.04
max,2026-08-01 00:00:00,18.10,66.15,13.09,10.56,3.23,6.03,1.28
std,NaN,1.81,9.30,3.96,1.09,0.32,0.70,0.29


In [75]:
base_final = base_final.rename(columns={"inad_cheque_especial":"cheque_especial",
                                    	"inad_cartao_rotativo": "cartao_rotativo",
                                        "inad_cartao_parcelado":"cartao_parcelado",
                                        "inad_consignado_privado":"consignado_privado",
                                        "inad_consignado_publico":"consignado_publico",
                                        "pf_total": "Inadimplência Total"})

In [121]:
fig = px.line(
    base_final,
    x="data",
    y=["cheque_especial", "cartao_rotativo", "cartao_parcelado", "consignado_privado", "consignado_publico", "Inadimplência Total"],
    title="Evolução Anual da Inadimplência por Modalidade",
    template="plotly_dark"
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)


fig

In [79]:
fig = px.line(
    base_final,
    x="data",
    y=["Inadimplência Total","selic_mensal"],
    title="Inadimplência Total vs. Selic (2011-2026)",
    template="plotly_dark"

)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig

In [122]:
ultimo = base_final[["cheque_especial", "cartao_rotativo", "cartao_parcelado", "consignado_privado", "consignado_publico", "Inadimplência Total"]].iloc[-1].sort_values()


fig = px.bar(
    x=ultimo.values,
    y=ultimo.index,
    orientation="h",
    text=ultimo.values,
    title=f"Ranking Inadimplência por Modalidade - Agosto de 2026",
    labels={"x": "Inadimplência (%)", "y":""},
    template="plotly_dark"
)

fig.update_traces(texttemplate="%{x:.2f}%", textposition="outside")
fig.update_layout(showlegend=False)


In [123]:
correlacoes = base_final[["cheque_especial", "cartao_rotativo", "cartao_parcelado", "consignado_privado", "consignado_publico", "Inadimplência Total","selic_mensal"]].corr()["selic_mensal"].drop("selic_mensal").sort_values()

fig = px.bar(
    x=correlacoes.values,
    y=correlacoes.index,
    orientation="h",
    text=correlacoes.values,
    color=correlacoes.values,
    color_continuous_scale=["#4ECDC4", "#FFFFFF", "#FF6B6B"],
    color_continuous_midpoint=0,
    title="Correlação com a Selic mensal",
    labels={"x": "Coeficiente de correlação", "y": ""}
)

fig.update_traces(texttemplate="%{x:.2f}", textposition="outside")
fig.update_layout(coloraxis_showscale=False, template="plotly_dark")


In [99]:
# Correlação com defasagem de 3, 6 e 9 meses
for lag in [0, 3, 6, 9, 12]:
    corr = base_final["selic_mensal"].shift(lag).corr(base_final["cartao_rotativo"])
    print(f"Defasagem {lag} meses: {corr:.2f}")

Defasagem 0 meses: 0.49
Defasagem 3 meses: 0.50
Defasagem 6 meses: 0.48
Defasagem 9 meses: 0.45
Defasagem 12 meses: 0.39
